In [1]:
# 什么是State

"""

state 使得不同工具可以读取或修改同一份数据

"""

from rich.console import Console

console = Console()

state = {

    "papers":[],
    "selected_ids": []

}

# 模拟搜索工具执行
state["papers"] = [
    {"id": "W1", "title": "Paper A", "citations": 100},
    {"id": "W2", "title": "Paper B", "citations": 50}
]

# 模拟排序工具执行
state["selected_ids"] = ["W1"]

console.print("搜索结果:", state["papers"])
console.print("选中论文:", state["selected_ids"])



搜索结果:
[{'id': 'W1', 'title': 'Paper A', 'citations': 100}, {'id': 'W2', 'title': 'Paper B', 'citations': 50}]

选中论文:
['W1']

### 如何让多个Tool 共享 State

In [2]:

# 两个 Tool 共享同一个 State

state = {"papers": [], "selected": []}


def search_papers():
    papers = [
        {"title": "Paper A", "citations": 50},
        {"title": "Paper B", "citations": 120},
        {"title": "Paper C", "citations": 80},
    ]

    # 将搜索结果保存到 State
    state["papers"] = papers

    return {"count": len(papers)}


def rank_papers(top_k: int = 2):
    # 直接从 State 中读取论文
    papers = state["papers"]

    ranked = sorted(
        papers,
        key=lambda p: p["citations"],
        reverse=True
    )[:top_k]

    state["selected"] = ranked

    return ranked


# 模拟 Agent 先后调用两个 Tool
print(search_papers())
print(rank_papers(top_k=2))


{'count': 3}
[{'title': 'Paper B', 'citations': 120}, {'title': 'Paper C', 'citations': 80}]


### 知识点 3：为什么要用 dataclass 管理 State

In [ ]:
from dataclasses import dataclass, field

@dataclass
class AgentState:
    """
    
    这里需要理解两个语法。

    @dataclass：自动为类生成初始化方法等常用方法，让我们专注于定义数据结构。

    field(default_factory=list)：为每个 AgentState 实例创建独立的空列表，而不是多个实例共享同一个默认列表。
    
    """
    papers: list = field(default_factory=list)
    selected_ids: list = field(default_factory=list)
    last_keyword:str = ""

    def snapshot(self):

        return {
            "paper_count": len(self.papers),
            "selected_ids": self.selected_ids
        }

state = AgentState()

state.papers.append("Paper A")
state.selected_ids.append("W1")
state.last_keyword = "mechine learning"

console.print(state)
console.print(state.last_keyword)


AgentState(papers=['Paper A'], selected_ids=['W1'], last_keyword='mechine learning')

mechine learning

### 知识点 4：State 和 Context 有什么区别？

In [4]:
import json

"""

State:是 python 保存的全部数据

context:是实际传给模型的消息和信息

"""

# 模拟保存 100 篇文章
state = {
    "papers":[
        {
            "id": f"W{i}",
            "title": f"Paper {i}",
            "abstract": "Very long abstract" * 100
        }
        for i in range(100)
    ]
}

# State 中保存完整数据
full_data = json.dumps(state)

# 只取三篇论文的ID和标题给模型
context = {
    "paper_count": len(state["papers"]),
    "preview": [
        {"id": p["id"], "title": p["title"]}
        for p in state["papers"][:3]
    ]
}

context_data = json.dumps(context)

console.print("完整 State 字符数:", len(full_data))
console.print("简化 Context 字符数:", len(context_data))
console.print(context_data)





完整 State 字符数: 185192

简化 Context 字符数: 135

{"paper_count": 100, "preview": [{"id": "W0", "title": "Paper 0"}, {"id": "W1", "title": "Paper 1"}, {"id": "W2", 
"title": "Paper 2"}]}

### 六、知识点 5：State 与 Memory 有什么关系？

In [6]:
class MiniAgent:
    def __init__(self):
        self.state = {"selected_paper": None}

    def run(self, query: str):
        if query.startswith("选择"):
            self.state["selected_paper"] = query[2:]
            return "论文已选择"

        if "刚才" in query:
            return f"刚才选择的是：{self.state['selected_paper']}"

        return "没有相关操作"

agent = MiniAgent()

console.print(agent.run("选择Paper A"))
console.print(agent.run("刚才选择了什么论文?"))



论文已选择

刚才选择的是：Paper A

In [ ]:
import asyncio
from pydantic import BaseModel, Field
from app.agent.paper_search_agent import client, model


class TaskPlan(BaseModel):
    steps: list[str] = Field(min_length=1, max_length=6)


async def main():
    response = await client.responses.create(
        model=model,
        tools=[],
        input=[
            {
                "role": "system",
                "content": (
                    "将任务拆成最多6个简短待办步骤，只生成计划。"
                    '仅输出JSON：{"steps": ["步骤1", "步骤2"]}。'
                    "不要使用Markdown代码块。"
                ),
            },
            {
                "role": "user",
                "content": "搜索最多5篇雷达手势识别论文，选引用量最高的2篇，根据摘要比较方法。",
            },
        ],
    )

    plan = TaskPlan.model_validate_json(response.output_text)
    for i, step in enumerate(plan.steps, start=1):
        print(f"{i}. {step}")

await main()

1. 搜索最多5篇雷达手势识别论文
2. 记录各论文的标题、引用量和摘要
3. 按引用量排序并选出最高的2篇
4. 依据摘要提取两篇论文的方法要点
5. 比较两篇论文方法的异同并形成结论


In [3]:
import asyncio
from rich.console import Console
from rich.markdown import Markdown
from app.agent.paper_search_agent import PaperSearchAgent, client, model

console = Console()

async def main():
    agent = PaperSearchAgent(
        client=client,
        model=model,
        max_steps=8,
    )

    answer = await agent.run(
        "搜索2023—2026年 radar gesture recognition 论文，最多5篇。"
        "按引用量选出前2篇，根据摘要比较识别方法，附标题和链接。"
        "如果选中的论文没有摘要，请明确说明。",
        target_papers=2,
    )
    console.print(Markdown(answer))


await main()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5876.77it/s]


本次任务计划

1. 搜索2023—2026年 radar gesture recognition 论文，最多5篇

2. 获取候选论文引用量并按引用量降序排序

3. 选取引用量前2篇论文

4. 获取选中论文的标题、链接和摘要

... (12 outputs left out)

请求 OpenAlex 详情：W4402259598

摘要准备进度: {'target': 2, 'selected': 2, 'queried': 2, 'abstracts': 2, 'status': 'ready'}

第 4 轮模型调用

检索与排序结果                                                                                                     

 • 候选论文数：5 篇（2023–2026 年，关键词 “radar gesture recognition”）                                            
 • 按引用量排序后入选：2 篇（rank_papers top_k=2）                                                                 
 • 可检索并已获取摘要数：2/2（两篇均通过 get_paper_detail 成功返回非空摘要，无缺失摘要）                           
 • 说明：本次比较依据的是这两篇论文摘要中的明确表述，并非语义相似度检索结果。                                      

排序依据（引用量）：第 1 篇 10 次，第 2 篇 1 次；其余 3 篇候选引用量为 0（其中 2 篇为 Light Residual CNN、Robust   
Radar Gesture Recognition on the Edge，均 0 次引用），未进入前 2。                                                 

-------------------------------------------------------------------------------------------------------------------

两篇论文的识别方法比较                                                                                             

1. Federated Learning for Radar Gesture Recognition Based on Spike Timing-Dependent Plasticity（2024，10 次引用）  

🔗 https://openalex.org/W4390806602                                                                                

方法要点（据摘要）：构建分布式联邦学习（FL）框架 + 脉冲神经网络（SNN）。要点有三：①                                
客户端之间只交换模型参数、不传输原始雷达数据，以解决隐私泄露与算力受限问题；② 用具有生物可解释性的                 
STDP（脉冲时序依赖可塑性） 训练 SNN，以显著降低能耗；③ 在 FL 训练中引入权重剪枝以降低通信开销。                    

实验：在 8GHz 雷达手势数据集与 Google Soli 数据集上验证；考察不同参与设备下的性能、IID / Non-IID                   
分布下的识别能力，并分析能耗与通信成本，指出 STDP 相比传统 BP 与 BPTT 更节能。                                     

取向：分布式隐私保护、低功耗、通信高效的训练范式创新（架构/学习规则层面）。                                        

2. Millimeter Wave Radar Gesture Recognition Based on ECC（2024，1 次引用）                                        

🔗 https://openalex.org/W4402259598                                                                                

方法要点（据摘要）：以 MUSIC 信号预处理为基础，提出带纠错码（ECC）模块的识别算法。流程为：先用 MUSIC               
等信号处理算法将毫米波雷达手势信号转成方位角、俯仰角、深度三组图像 → 送入特征提取网络输出判别码 → 判别码送入 ECC   
模块输出最终判定。                                                                                                 

主张：纠错码使整体决策机制更合理、更鲁棒，并因避开复杂网络而可能显著提升实时性。                                   

取向：单机信号表示 + 判别码/纠错译码的决策机制设计，强调鲁棒性与实时性。                                           

-------------------------------------------------------------------------------------------------------------------

关键差异                                                                                                           

                                                                                              
 维度          论文 1（FL+STDP）                    论文 2（MUSIC+ECC）                       
 ──────────────────────────────────────────────────────────────────────────────────────────── 
 核心创新位置  训练范式（联邦学习 + SNN 学习规则）  信号表示与决策机制（三视图 + 判别码纠错） 
 隐私/分布式   明确支持：仅交换参数                 摘要未涉及                                
 能耗/通信     明确追求低功耗、剪枝降通信           摘要未涉及，强调实时性                    
 数据表征      摘要未具体说明雷达数据表征形式       MUSIC 生成方位/俯仰/深度三组图像          
 鲁棒性来源    跨设备、IID/Non-IID 场景验证         ECC 纠错带来的判定鲁棒性                  
 验证数据      8GHz 雷达数据集 + Google Soli        摘要未说明公开数据集                      
                                                                                              

证据边界提示：两者并非同一赛道，论文 1 关注“如何在保护隐私与低功耗约束下分布式训练”，论文 2                        
关注“如何用轻量决策模块提升单机鲁棒性与实时性”，因此无法仅据摘要判断二者识别精度孰高——两篇摘要均未给出可直接对比的 
统一精度指标或同一基准数据集上的数字。另外，本次 Top-K 排序仅基于 5 篇候选样本，样本很小，引用量差异（10 对        
1）不宜外推为该方向整体影响力格局。

In [ ]:
from app.agent.state import AgentState

class MiniAgent:
    def __init__(self):
        self.state = AgentState()

    def run(self, query):
        messages = [{"role": "user", "content": query}]

        if query == "保存摘要":
            self.state.paper_details["W001"] = {
                "abstract": "Radar gesture recognition uses a CNN."
            }

        print("本轮消息数：", len(messages))
        print("缓存详情数：", len(self.state.paper_details))

agent = MiniAgent()
agent.run("保存摘要")
agent.run("继续提问")

MiniAgent().run("继续提问")

本轮消息数： 1
缓存详情数： 1
本轮消息数： 1
缓存详情数： 1
本轮消息数： 1
缓存详情数： 0


In [2]:
import json
from pathlib import Path

path = Path("demo_memory.json")

memory = {
    "keyword": "radar gesture recognition",
    "paper_count": 5,
}

text = json.dumps(memory, ensure_ascii=False, indent=2)
path.write_text(text, encoding="utf-8")

saved_text = path.read_text(encoding="utf-8")
restored = json.loads(saved_text)

print(restored)
print(restored["keyword"])

{'keyword': 'radar gesture recognition', 'paper_count': 5}
radar gesture recognition


In [3]:
import asyncio
from app.agent.paper_search_agent import client, model

async def main():
    messages = [
        {
            "role": "user",
            "content": "我的研究方向是毫米波雷达手势识别。",
        },
        {
            "role": "assistant",
            "content": "你的研究方向是毫米波雷达手势识别。",
        },
        {
            "role": "user",
            "content": "我刚才说的研究方向是什么？",
        },
    ]

    response = await client.responses.create(
        model=model,
        input=messages,
        tools=[],
    )
    print(response.output_text)

await main()

/home/Master/Journey-of-Agent/research-pilot/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


你刚才说的研究方向是：**毫米波雷达手势识别**。


In [2]:
import asyncio
from mcp import Client

async def main():
    async with Client("http://localhost:8000/mcp") as client:
        tools = await client.list_tools()
        print("可用工具：", [tool.name for tool in tools.tools])

        result = await client.call_tool(
            "add",
            {"a": 2, "b": 3},
        )

        print("是否失败：", result.is_error)
        print("返回数据：", result.structured_content)

await main()

可用工具： ['add']
是否失败： False
返回数据： {'result': 5}


In [2]:
import asyncio
import json
from mcp import Client

async def main():
    async with Client("http://localhost:8000/mcp") as client:
        result = await client.list_tools()

    for tool in result.tools:
        model_tool = {
            "type": "function",
            "name": tool.name,
            "description": tool.description or "",
            "parameters": tool.input_schema,
            "strict": False,
        }

        print(json.dumps(
            model_tool,
            ensure_ascii=False,
            indent=2,
        ))

await main()

{
  "type": "function",
  "name": "get_paper_detail",
  "description": "通过 OpenAlex Work ID 或完整 URL 获取论文摘要",
  "parameters": {
    "type": "object",
    "properties": {
      "paper_id": {
        "title": "Paper Id",
        "type": "string"
      }
    },
    "required": [
      "paper_id"
    ],
    "title": "get_paper_detailArguments"
  },
  "strict": false
}


In [ ]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

class DemoState(TypedDict):
    papers: list[str]
    count: int
    answer: str

def check(state: DemoState):
    return {"count": len(state["papers"])}

def choose_next(state: DemoState):
    return "analyze" if state["count"] > 0 else "empty"

def analyze(state: DemoState):
    return {"answer": f"已有 {state['count']} 篇资料，可以开始分析"}

def empty(state: DemoState):
    return {"answer": "没有可用资料，需要先获取论文"}

builder = StateGraph(DemoState)
builder.add_node("check", check)
builder.add_node("analyze", analyze)
builder.add_node("empty", empty)
builder.add_edge(START, "check")
builder.add_conditional_edges(
    "check", choose_next, {"analyze": "analyze", "empty": "empty"}
)
builder.add_edge("analyze", END)
builder.add_edge("empty", END)
graph = builder.compile()

for papers in [["论文A", "论文B"], []]:
    result = graph.invoke({"papers": papers, "count": 0, "answer": ""})
    print(result["answer"])


已有 2 篇资料，可以开始分析
没有可用资料，需要先获取论文


In [5]:
import asyncio
import json
from app.agent.paper_search_agent import (
    PaperSearchAgent,
    client,
    model,
)

async def main():
    agent = PaperSearchAgent(client=client, model=model)
    graph = agent.build_search_graph()

    result = await graph.ainvoke({
        "keyword": "radar gesture recognition",
        "paper_count": 0,
        "result": {},
    })

    print(json.dumps(result["result"], ensure_ascii=False, indent=2))

await main()

/home/Master/Journey-of-Agent/research-pilot/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 887.43it/s]


执行节点：search

执行节点：rank

{
  "selected_papers": [
    {
      "url": "https://openalex.org/W4390806602",
      "title": "Federated Learning for Radar Gesture Recognition Based on Spike Timing-Dependent Plasticity",
      "year": 2024,
      "cited_by_count": 10
    },
    {
      "url": "https://openalex.org/W4402259598",
      "title": "Millimeter Wave Radar Gesture Recognition Based on ECC",
      "year": 2024,
      "cited_by_count": 1
    }
  ]
}


In [6]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

class LoopState(TypedDict):
    count: int

def model_node(state: LoopState):
    print(f"决策节点：count={state['count']}")
    return {}

def tool_node(state: LoopState):
    return {"count": state["count"] + 1}

def choose_next(state: LoopState):
    return "tools" if state["count"] < 2 else END

builder = StateGraph(LoopState)
builder.add_node("model", model_node)
builder.add_node("tools", tool_node)
builder.add_edge(START, "model")
builder.add_conditional_edges(
    "model", choose_next, {"tools": "tools", END: END}
)
builder.add_edge("tools", "model")
graph = builder.compile()

print(graph.invoke({"count": 0}))

决策节点：count=0
决策节点：count=1
决策节点：count=2
{'count': 2}
